# Pre-processing and Training Data Development

In [3]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the Instagram Analytics Dataset 
insta_data = pd.read_csv('/Users/rupashree/Downloads/Instagram_Analytics.csv')

In [4]:
# Count missing values in any columns 
missing_values = insta_data.isna().sum()
print(missing_values)
# Drop Duplicate rows
insta_df = insta_data.drop_duplicates()
insta_df.shape

post_id             0
upload_date         0
media_type          0
likes               0
comments            0
shares              0
saves               0
reach               0
impressions         0
caption_length      0
hashtags_count      0
followers_gained    0
traffic_source      0
engagement_rate     0
content_category    0
dtype: int64


(29999, 15)

In [5]:
# Convert upload_date to datetime column 
insta_df['upload_date'] = pd.to_datetime(insta_df['upload_date'])
# Use insta data to select the columns of dtype object 
obj_col = ['post_id', 'media_type', 'traffic_source', 'content_category']
insta_df[obj_col] = insta_df[obj_col].astype('category')
insta_df.dtypes

post_id                   category
upload_date         datetime64[ns]
media_type                category
likes                        int64
comments                     int64
shares                       int64
saves                        int64
reach                        int64
impressions                  int64
caption_length               int64
hashtags_count               int64
followers_gained             int64
traffic_source            category
engagement_rate            float64
content_category          category
dtype: object

In [8]:
# Set constraint to remove illogical impossible value in instagram dataset
insta_df['invalid_metrics_flag'] = (
    (insta_df['likes'] > insta_df['reach']) |
    (insta_df['comments'] > insta_df['reach']) |
    (insta_df['saves'] > insta_df['reach']) |
    (insta_df['impressions'] < insta_df['reach'])
)
flagged_rows = insta_df[insta_df['invalid_metrics_flag']]
print(flagged_rows.head())
flagged_rows.shape

      post_id                upload_date media_type   likes  comments  shares  \
4   IG0000005 2025-03-21 09:25:22.954916      Video   99646      2703    4444   
18  IG0000019 2025-06-20 09:25:22.954916      Photo  111231      6343     209   
21  IG0000022 2025-10-04 09:25:22.954916      Photo  179760      1745    4286   
25  IG0000026 2025-02-18 09:25:22.954916      Video   77071      2115    2268   
32  IG0000033 2025-04-17 09:25:22.954916       Reel  166153       994    4272   

    saves  reach  impressions  caption_length  hashtags_count  \
4    9746   7004       495406            1905               8   
18  14970  61120        85737            2062              27   
21  13818  57656       115755             401               2   
25   5573   7054        26092            1067              29   
32   3066  30495       138675             984              21   

    followers_gained traffic_source  engagement_rate content_category  \
4                155        Profile            23

(1552, 16)

In [9]:
# Keep only valid rows
clean_insta_df = insta_df.loc[~insta_df['invalid_metrics_flag']].copy()
clean_insta_df.head()
clean_insta_df.shape

(28447, 16)

In [13]:
# Define target 
y = clean_insta_df['engagement_rate']

# Drop unnecessary columns
X = clean_insta_df.drop(['engagement_rate', 'post_id','upload_date'], axis = 1)
print(X.shape)

(28447, 13)


In [14]:
# One-hot encoding
X = pd.get_dummies(X, drop_first = True)
print(X.shape)
X.head()

(28447, 27)


,likes,comments,shares,saves,reach,impressions,caption_length,hashtags_count,followers_gained,invalid_metrics_flag,...,traffic_source_Reels Feed,content_category_Comedy,content_category_Fashion,content_category_Fitness,content_category_Food,content_category_Lifestyle,content_category_Music,content_category_Photography,content_category_Technology,content_category_Travel
0,31627,7559,4530,6393,615036,1007750,1340,3,899,False,...,False,False,False,False,False,False,False,False,True,False
1,63206,3490,1680,6809,1237071,1345900,1351,20,805,False,...,False,False,False,True,False,False,False,False,False,False
2,94373,3727,1761,8367,1127470,1305369,242,24,758,False,...,True,False,False,False,False,False,False,False,False,False
3,172053,7222,2875,9290,764030,897874,446,11,402,False,...,False,False,False,False,False,False,True,False,False,False
5,10500,7337,3601,5163,311157,381943,1448,12,904,False,...,True,False,False,False,False,False,False,True,False,False


In [16]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (22757, 27)
Test: (5690, 27)


In [17]:
# Scale Numeric features only 
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit only on training data 
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)